# 04 — Model Comparison: Dense vs Conv2D vs LSTM

This notebook loads the results saved by notebooks 01–03 and produces a comprehensive comparison.

> ⚠️ **Run notebooks 01, 02, and 03 first** to generate the results files.

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

sns.set_theme(style='whitegrid', palette='muted')
COLORS = ['#4C72B0', '#55A868', '#C44E52']

## 1. Load Results

In [ ]:
files = {
    'Dense':  '../results/dense_results.json',
    'Conv2D': '../results/conv2d_results.json',
    'LSTM':   '../results/lstm_results.json',
}

data = {}
for name, path in files.items():
    with open(path) as f:
        data[name] = json.load(f)

print('Loaded results for:', list(data.keys()))

## 2. Summary Table

In [ ]:
rows = []
for name, d in data.items():
    rows.append({
        'Model':          name,
        'Test Accuracy':  f"{d['test_accuracy']:.4f}",
        'Test Loss':      f"{d['test_loss']:.4f}",
        'Total Params':   f"{d['total_params']:,}",
        'Training Time':  f"{d['training_time_s']}s",
    })

df = pd.DataFrame(rows)
df = df.set_index('Model')
print(df.to_string())

## 3. Accuracy Comparison Bar Chart

In [ ]:
names = list(data.keys())
accs  = [data[n]['test_accuracy'] for n in names]

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(names, accs, color=COLORS, width=0.5, edgecolor='white', linewidth=1.5)

for bar, acc in zip(bars, accs):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() - 0.005,
            f'{acc:.4f}', ha='center', va='top', color='white', fontweight='bold', fontsize=12)

ax.set_ylim(0.97, 1.0)
ax.set_title('Test Accuracy by Model', fontsize=15, fontweight='bold')
ax.set_ylabel('Test Accuracy')
ax.set_xlabel('Architecture')
plt.tight_layout()
plt.show()

## 4. Parameter Count vs Accuracy (Efficiency Plot)

In [ ]:
params = [data[n]['total_params'] for n in names]
times  = [data[n]['training_time_s'] for n in names]

fig, ax = plt.subplots(figsize=(8, 5))

for i, (name, param, acc) in enumerate(zip(names, params, accs)):
    ax.scatter(param, acc, s=300, color=COLORS[i], zorder=5, label=name)
    ax.annotate(name, (param, acc), textcoords='offset points',
                xytext=(10, 5), fontsize=11, color=COLORS[i], fontweight='bold')

ax.set_xlabel('Total Parameters', fontsize=12)
ax.set_ylabel('Test Accuracy', fontsize=12)
ax.set_title('Efficiency: Accuracy vs Parameter Count', fontsize=13, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.4)
plt.tight_layout()
plt.show()

## 5. Training Curves — All Models

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for i, (name, d) in enumerate(data.items()):
    epochs = range(1, len(d['history']['accuracy']) + 1)
    axes[0].plot(epochs, d['history']['val_accuracy'], label=name, color=COLORS[i], linewidth=2)
    axes[1].plot(epochs, d['history']['val_loss'],     label=name, color=COLORS[i], linewidth=2)

axes[0].set_title('Validation Accuracy over Epochs', fontsize=13)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()
axes[0].grid(True, alpha=0.4)

axes[1].set_title('Validation Loss over Epochs', fontsize=13)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()
axes[1].grid(True, alpha=0.4)

plt.suptitle('All Models — Validation Metrics', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 6. Training Time Comparison

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.barh(names[::-1], times[::-1], color=COLORS[::-1], edgecolor='white', linewidth=1.5)

for bar, t in zip(bars, times[::-1]):
    ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height() / 2,
            f'{t}s', va='center', fontweight='bold', fontsize=11)

ax.set_xlabel('Training Time (seconds)')
ax.set_title('Total Training Time per Model (10 epochs)', fontsize=13, fontweight='bold')
ax.grid(True, axis='x', alpha=0.4)
plt.tight_layout()
plt.show()

## 7. Key Takeaways

| | Dense | Conv2D | LSTM |
|---|---|---|---|
| **Best for** | Baselines, tabular | Image tasks | Sequential data |
| **Accuracy** | Good | ✅ Best | Very good |
| **Params** | High | ✅ Lowest | Medium |
| **Speed** | ✅ Fastest | Medium | Slowest |
| **Interprets images as** | Flat vector | Spatial grid | Row sequence |

### Conclusions
- **Conv2D** achieves the best accuracy with the fewest parameters — it's the right tool for image classification because it understands spatial relationships.
- **Dense** is surprisingly competitive and very fast, but it treats every pixel independently (no spatial awareness).
- **LSTM** is not the natural fit for images, but it still performs well by treating images as a sequence of rows. It's the slowest due to sequential computation.